# 16 - Uso de Word2Vec Preentrenado
 
Goal: Ejemplo de uso de embeddings Word2Vec preentrenados en un modelo Keras simple.
 
Run with: conda activate tfenv

In [81]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import plotly.express as px
import pandas as pd
import time

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.21.0


In [94]:
# Importar embeddings Word2Vec entrenados previamente
class MyWord2Vec():
	def __init__(self, path='myWord2Vec/'):
		super(MyWord2Vec, self).__init__()
		target_embeddings = np.load(path + 'target_embeddings.npy')
		context_embeddings = np.load(path + 'context_embeddings.npy')
		text_vocab = np.load(path + 'text_vocab.npy', allow_pickle=True).item()
		self.layer = layers.Embedding(
			input_dim=target_embeddings.shape[0],
			output_dim=target_embeddings.shape[1],
			weights=[target_embeddings],
			trainable=False,  # O True si quieres ajustar
			name='pretrained_embedding'
		)
		self.target_embeddings = target_embeddings
		self.context_embeddings = context_embeddings
		self.text_vocab = text_vocab
		self.final_embeddings = (target_embeddings + context_embeddings) / 2
		print('Embeddings cargados:', target_embeddings.shape, context_embeddings.shape, 'Vocabulario cargado:', len(text_vocab))

	def __call__(self, token):
		token_id = self.text_vocab.get(token)
		if token_id is None:
			raise ValueError(f"Token '{token}' no encontrado en el vocabulario.")
		return self.final_embeddings[token_id]

myWord2Vec = MyWord2Vec()

Embeddings cargados: (807, 64) (807, 64) Vocabulario cargado: 807


In [98]:
myWord2Vec('bank')

array([-0.28501534, -0.5073903 ,  0.3754411 , -0.16484009, -0.1826826 ,
       -0.5811516 ,  0.25318754, -0.20066723, -0.03234106, -0.1983294 ,
        0.22193125, -0.07133868, -1.1277426 , -0.14788602,  0.00594485,
        0.06697989,  0.2680027 ,  0.30610293,  0.02022836, -0.14936   ,
       -0.35025677,  0.9948102 , -0.29224348,  0.27824396, -0.06160453,
        0.96823585, -1.0880908 ,  0.20591159,  0.35358378,  0.03991617,
       -0.0276342 ,  0.5968418 ,  0.7929538 ,  0.74884087,  0.40281463,
        0.12895238,  0.45341355, -1.1052495 , -0.4245516 ,  0.03584774,
        0.6724809 ,  0.08946411, -0.08444281, -0.40274462,  0.22121572,
       -1.2098794 ,  0.04220062,  0.5373121 , -0.3225735 ,  0.08055313,
       -0.16905993, -0.16546796, -0.26491246, -0.31321383,  0.10717484,
       -0.14396463, -0.86192745,  0.5560495 ,  0.08061355, -0.07986355,
        0.5029609 , -0.38039207,  0.655138  ,  0.38013253], dtype=float32)

In [ ]:
# Ejemplo: Clasificación de palabras (dummy) usando embeddings preentrenados

# Modelo simple: toma el embedding y lo pasa por una capa densa
model = keras.Sequential([
	layers.Input(shape=(), dtype='int32'),
	myWord2Vec.layer,  # Capa de embedding preentrenada
	layers.Flatten(),
	layers.Dense(8, activation='relu'),
	layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

train_ds = tf.data.Dataset.from_tensor_slices((insert dummy data here))

history = model.fit(
    train_ds,
    epochs=20,
    verbose=1
)

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ pretrained_embedding            │ (None, 64)             │        51,648 │
│ (Embedding)                     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 8)              │           520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 52,177 (203.82 KB)

 Trainable params: 529 (2.07 KB)

 Non-trainable params: 51,648 (201.75 KB)

NameError: name 'X' is not defined